# Selezione delle coppie: metodo della distanza sui dati giornalieri

**Obiettivo.** Trovare 6-9 coppie di azioni dello stesso settore dell'S&P 500, usando solo i dati giornalieri. Sulle coppie scelte il progetto farà poi addestramento e test sui dati intraday (barre da 1 minuto).

**Come è organizzato il notebook**

| Parte | Cosa fa | Ruolo |
|---|---|---|
| 1-4 | Parametri, dati, filtri sui titoli, settore GICS | preparazione |
| 5 | **Metodo della distanza** → lista finale | **canale principale** |
| 6 | Diagnostiche statistiche delle coppie scelte | controlli, non filtri |
| 7 | Test di cointegrazione sulle coppie dello stesso settore | canale di confronto |
| 8 | Coppie "da manuale" (candidate teoriche) | controllo di buon senso |
| 8b | Stessa regola su 10, 15 e 20 anni | controllo di robustezza |
| 9 | Riepilogo, limiti, salvataggio | per il report |

**Regola anti data-snooping.** I dati giornalieri usati finiscono il **15 giugno 2026**, il giorno prima dell'inizio dei dati intraday. Così la scelta delle coppie non "vede" il periodo di test. Le soglie sono nella cella dei parametri e vanno fissate *prima* di guardare i risultati: ritoccarle finché esce la coppia che ci piace vorrebbe dire adattare la scelta ai dati (data snooping), e il test perderebbe valore (Lo e MacKinlay, 1990; Harvey, Liu e Zhu, 2016).

**Mini-glossario**
- **Spread**: la differenza tra i prezzi (o i rendimenti cumulati) dei due titoli di una coppia. La strategia scommette che, quando si allarga, poi si restringe.
- **SSD** (somma dei quadrati delle distanze): numero che misura quanto i due titoli sono rimasti vicini nel periodo. Più è basso, più si sono mossi insieme.
- **Correlazione dei rendimenti**: quanto i due titoli salgono e scendono insieme giorno per giorno (da -1 a +1).
- **Cointegrazione**: proprietà statistica per cui la distanza tra due prezzi non si allontana all'infinito ma torna verso una media (Engle e Granger, 1987).
- **Half-life**: tempo che lo spread impiega a tornare a metà strada verso la sua media.
- **FDR** (false discovery rate): correzione per il fatto che, provando migliaia di coppie, alcune risultano "buone" per solo caso (Benjamini e Hochberg, 1995).

## 1. Parametri, con il perché di ogni scelta

Legenda: **[ACC]** scelta con riferimento accademico, **[ECO]** scelta con motivazione economica, **[NOSTRA]** scelta nostra, senza fonte: va dichiarata come tale nel report.

In [43]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.stattools import coint
from statsmodels.stats.multitest import multipletests
from data import load

# ---- Periodo di selezione -------------------------------------------------------------------------
SEL_START = "2021-06-16"
SEL_END   = "2026-06-15"
# [NOSTRA/ECO] 5 anni. Gatev et al. (2006) formano le coppie su 12 mesi; noi usiamo piu storia perche con
#   pochi mesi la classifica per distanza e molto sensibile al caso. Il rovescio della medaglia: su 5 anni
#   un legame tra due aziende puo cambiare (fusioni, cambi di strategia). Per questo controlliamo la
#   stabilita nelle due meta del periodo (parte 6). Si parte dal giugno 2021 per lasciare fuori lo shock del
#   Covid (marzo 2020), che sposta tutti i prezzi insieme e falsa le distanze.
# [ACC/ECO] Si finisce il 15 giugno 2026: i dati intraday iniziano il 16, e la selezione non deve vedere il test.

# ---- Filtri sul singolo titolo --------------------------------------------------------------------
MIN_DOLLAR_VOLUME = 50e6
MAX_MISSING_PCT   = 0.01
# [ECO] Si opera in giornata aprendo e chiudendo 4 operazioni per trade: servono titoli dove si compra e si
#   vende senza muovere il prezzo. Il valore scambiato (prezzo x volume, mediana giornaliera) e piu
#   significativo del solo volume. 50 milioni di dollari e un numero tondo scelto da noi [NOSTRA]: non c'e
#   una soglia "ufficiale". Dati completi: l'indice cumulato della distanza parte dal primo giorno, quindi
#   un titolo senza storia dal 2021 non e confrontabile.

# ---- Industria ------------------------------------------------------------------------------------
GICS_LEVEL = "industry_group"
# [ECO] Due aziende dello stesso gruppo industriale hanno clienti, costi e normativa simili, quindi sono
#   esposte agli stessi fattori. E il motivo per cui il loro prezzo relativo dovrebbe tornare normale
#   (idea del "common trends model" in Vidyamurthy, 2004). Il livello "sub_industry" sarebbe piu stretto
#   (124 gruppi) ma lascia troppo poche coppie; "industry_group" ha 25 gruppi.

# ---- Metodo della distanza: regole di selezione ----------------------------------------------------
MAX_CORR_SAME_COMPANY = 0.97
# [ECO] Sopra 0.97 sono quasi la stessa azienda (es. GOOG e GOOGL, due classi di azioni). Lo spread e
#   minuscolo: i costi di 4 operazioni lo cancellano. Le coppie quasi identiche sono da evitare anche nel
#   testo del programma (es. SPY-VOO).
MIN_CORR = 0.60
# [NOSTRA] Soglia minima di correlazione dei rendimenti giornalieri. Non e un criterio di Gatev et al., che
#   usano solo la distanza. Serve perche la distanza da sola, su prezzi cumulati, puo premiare due titoli
#   che hanno solo fatto percorsi simili senza muoversi insieme giorno per giorno. Per una strategia
#   intraday contano i movimenti giornalieri congiunti piu del livello dei prezzi. 0.6 e un pavimento di
#   buon senso, volutamente basso: la selezione la fa la distanza.
N_PAIRS_FINAL = 9
MAX_PER_INDUSTRY = 3
# [ECO] Da 6 a 9 coppie e il numero indicato nel programma. Al massimo 3 per industria per non dipendere da
#   un solo settore (rischio comune a tutte le coppie dello stesso settore). Ogni titolo compare in una
#   sola coppia [NOSTRA]: cosi i trade delle coppie sono piu indipendenti tra loro.

ADDED_PAIRS = [("KO", "PEP")]
# [NOSTRA] Coppie aggiunte per decisione del gruppo, FUORI dalla regola di selezione. KO-PEP (Coca-Cola e Pepsi)
#   e l'esempio classico di pair trading e ha una chiara giustificazione economica (stesso mercato delle bevande
#   analcoliche, stessi clienti e canali di vendita). Nei dati del periodo pero non e tra le coppie piu vicine
#   (posto 240 su 2.390 per SSD) e non risulta cointegrata. Per questo e contrassegnata come "aggiunta per scelta"
#   (colonna `origine`) e nel report va presentata e valutata a parte, per non far passare per risultato della
#   regola una decisione presa a priori. Con questa aggiunta le coppie sono 10, oltre le 6-9 del programma.
# ---- Test statistici (parti 6 e 7: diagnostiche, non filtri del canale principale) -----------------
COINT_P      = 0.05      # [ACC] p-value classico del test di Engle-Granger (1987)
BETA_RANGE   = (0.5, 2.0)  # [NOSTRA] un hedge ratio estremo vuol dire che un titolo domina l'altro
HL_RANGE     = (1, 30)   # [NOSTRA] half-life giornaliera in giorni; vedi la spiegazione nella parte 6
MIN_CROSSINGS_PER_YEAR = 12  # [NOSTRA] almeno una volta al mese lo spread torna alla media: servono molti trade
STAB_P       = 0.10      # [NOSTRA] p-value in ciascuna meta del periodo (2.5 anni l'una: meno potenza, soglia piu larga)
STAB_BETA_TOL = 0.30     # [NOSTRA] l'hedge ratio non deve cambiare piu del 30% tra le due meta
MIN_CORR_COINT = 0.70    # [NOSTRA] solo per il canale di confronto della parte 7
FDR_ALPHA    = 0.10      # [ACC] Benjamini-Hochberg; 10% e un valore usuale quando si cercano candidate, non conferme

# ---- Robustezza al periodo (parte 8b) ---------------------------------------------------------------
ROBUSTNESS_STARTS = {"5 anni": SEL_START, "10 anni": "2016-06-16", "15 anni": "2011-06-16", "20 anni": "2006-11-02"}
# [NOSTRA] Stessa regola, stessa data di fine, ma inizio piu indietro. NON serve a scegliere il periodo (la lista resta
#   quella dei 5 anni): serve a vedere quali coppie restano vicine anche in periodi diversi. Una coppia che compare
#   in piu finestre ha un legame meno legato a un singolo periodo. Si dichiara nel report come controllo di robustezza.

## 2. Dati giornalieri nel periodo di selezione

Si usa il prezzo di chiusura aggiustato per split e dividendi (`PX_LAST`): senza l'aggiustamento un frazionamento di azioni sembrerebbe un crollo del prezzo e falserebbe distanze e correlazioni. I rendimenti vengono dalla colonna `ret`, calcolata solo tra giorni di borsa consecutivi (alcune serie hanno buchi di anni).

In [44]:
daily = load("daily")
daily["ticker"] = daily["ticker"].astype(str)
daily = daily[(daily["date"] >= SEL_START) & (daily["date"] <= SEL_END)]

prices  = daily.pivot(index="date", columns="ticker", values="PX_LAST")
volume  = daily.pivot(index="date", columns="ticker", values="PX_VOLUME")
returns = daily.pivot(index="date", columns="ticker", values="ret")
prices_all, returns_all = prices.ffill(), returns      # universo completo, serve per le coppie teoriche

years = len(prices) / 252
print(f"{len(prices)} giorni di borsa ({years:.1f} anni), {prices.shape[1]} ticker, dal {prices.index.min().date()} al {prices.index.max().date()}")
funnel = {"titoli di partenza": prices.shape[1]}

1255 giorni di borsa (5.0 anni), 501 ticker, dal 2021-06-16 al 2026-06-15


## 3. Filtri sul singolo titolo

Perché: una coppia è tradabile solo se *entrambi* i titoli sono liquidi e hanno dati completi. Il survivorship bias (l'universo sono i titoli dell'indice di oggi) resta un limite da dichiarare.

In [45]:
dollar_volume = (prices * volume).median()
missing = prices.isna().mean()
liquid = (dollar_volume >= MIN_DOLLAR_VOLUME) & (missing <= MAX_MISSING_PCT)
funnel["liquidi e con pochi buchi"] = int(liquid.sum())

prices = prices.loc[:, liquid].ffill()            # i pochi buchi si riempiono con l'ultimo prezzo noto (mai con prezzi futuri)
returns = returns.loc[:, liquid]

complete = prices.columns[prices.notna().all()]   # servono prezzi dal primo all'ultimo giorno
prices, returns = prices[complete], returns[complete]
funnel["con dati completi nel periodo"] = len(complete)
print(f"{liquid.sum()} titoli liquidi, {len(complete)} con storia completa dal {prices.index[0].date()}")

307 titoli liquidi, 307 con storia completa dal 2021-06-16


## 4. Settore GICS

`data/gics.csv` (da Bloomberg) ha le colonne `ticker` (senza suffisso, es. `KO`), `industry_group` e `sub_industry`. **Limite da dichiarare:** Bloomberg dà la classificazione *attuale*: la riclassificazione del 2018 ha spostato alcune aziende e il file non ha la storia.

In [46]:
g = pd.read_csv(Path("../data/gics.csv"))
short = {t.split()[0]: t for t in prices_all.columns}      # "KO" -> "KO UN Equity"
gics = pd.Series({short[t]: v for t, v in zip(g["ticker"], g[GICS_LEVEL]) if t in short})
print(f"GICS ({GICS_LEVEL}): {gics.reindex(prices.columns).notna().sum()} dei {prices.shape[1]} titoli, {gics.nunique()} gruppi")
gics.reindex(prices.columns).value_counts()

GICS (industry_group): 307 dei 307 titoli, 25 gruppi


Beni strumentali                  29
Servizi finanziari                28
Attrezzature e servizi sanitar    23
Software e servizi                20
Semiconduttori e apparecchiatu    19
Farmaceutica, Biotecnologie e     18
Energia                           17
Tecnologia hardware e apparecc    13
Servizi al consumatore            13
Assicurazioni                     13
Servizi di pubblica utilità       12
Materiali                         12
Beni discrezionali Distribuzio    12
Alimenti, bevande e tabacco       11
Fondi investimento immobiliare    10
Trasporto                         10
Servizi commerciali e professi     8
Media e intrattenimento            8
Banche                             8
Beni di consumo Distribuzione      6
Prodotti durevoli e tessili        6
Servizi di telecomunicazione       4
Prodotti per la casa e l'igien     3
Automezzi e componenti             3
Gestione e sviluppo immobiliar     1
Name: count, dtype: int64

## 5. Canale principale: metodo della distanza

**Idea** (Gatev, Goetzmann e Rouwenhorst, 2006, *Review of Financial Studies*): se due titoli sono stati "vicini" nel passato, probabilmente lo restano; quando si allontanano, si scommette sul ritorno. Il metodo non richiede di assumere un modello statistico (come la cointegrazione): usa solo la distanza osservata, ed è per questo il riferimento della letteratura sul pair trading (rassegna in Krauss, 2017).

**Passi**
1. Ogni prezzo viene diviso per il suo primo valore: parte da 1 ed è il valore di 1 dollaro investito all'inizio del periodo. Così titoli con prezzi diversi (10 $ contro 500 $) diventano confrontabili.
2. Per ogni coppia nello stesso gruppo industriale si calcola la SSD: la somma, su tutti i giorni, del quadrato della differenza tra i due indici cumulati.
3. Si scartano le coppie quasi identiche e quelle con correlazione dei rendimenti troppo bassa.
4. Si ordina per SSD crescente e si prendono le prime, un titolo in una sola coppia, con il tetto per industria.

**Perché questo è coerente con il nostro progetto:** l'indice cumulato presuppone di investire lo *stesso capitale* nei due titoli. È la stessa scelta che abbiamo fatto per le posizioni (importo uguale sulle due gambe, senza leva).

In [47]:
norm = prices / prices.iloc[0]
names = list(norm.columns)

x = norm.values.T                                  # titoli x giorni
sq = (x ** 2).sum(axis=1)
SSD = sq[:, None] + sq[None, :] - 2 * x @ x.T      # SSD tra tutte le coppie: sum((xi - xj)^2) = sum xi^2 + sum xj^2 - 2 xi.xj

iu = np.triu_indices(len(names), k=1)
ind = gics.reindex(names).values
C = returns.corr().reindex(index=names, columns=names).values

pairs = pd.DataFrame({"A": [names[i] for i in iu[0]], "B": [names[j] for j in iu[1]], "ssd": SSD[iu], "corr": C[iu],
                      "industry": ind[iu[0]], "same_industry": ind[iu[0]] == ind[iu[1]]})
dist_funnel = {"coppie possibili (tutti i titoli)": len(pairs)}

same = pairs[pairs["same_industry"]].sort_values("ssd").reset_index(drop=True)
same["rank_ssd"] = np.arange(1, len(same) + 1)               # posto nella classifica delle coppie dello stesso gruppo
dist_funnel["stessa industria"] = len(same)

identical = same[same["corr"] > MAX_CORR_SAME_COMPANY]
cand = same[same["corr"] <= MAX_CORR_SAME_COMPANY]
dist_funnel[f"escluse quasi identiche (corr > {MAX_CORR_SAME_COMPANY})"] = len(cand)
print("Escluse come quasi identiche:"); display(identical[["A", "B", "corr", "ssd"]].round(4))

cand = cand[cand["corr"] >= MIN_CORR]
dist_funnel[f"correlazione >= {MIN_CORR}"] = len(cand)

Escluse come quasi identiche:


,A,B,corr,ssd
1,GOOG UW Equity,GOOGL UW Equity,0.9957,3.8615


In [48]:
# selezione: classifica per SSD, un titolo per coppia, tetto per industria
chosen, used, per_ind = [], set(), {}
for _, row in cand.sort_values("ssd").iterrows():
    a, b, industry = row["A"], row["B"], row["industry"]
    if a in used or b in used or per_ind.get(industry, 0) >= MAX_PER_INDUSTRY:
        continue
    chosen.append(row); used.update([a, b]); per_ind[industry] = per_ind.get(industry, 0) + 1
    if len(chosen) == N_PAIRS_FINAL:
        break
final = pd.DataFrame(chosen).reset_index(drop=True)
final["origine"] = "regola"
dist_funnel["coppie finali (dalla regola)"] = len(final)

# coppie aggiunte per scelta (ADDED_PAIRS): fuori dalla regola, si accodano e restano contrassegnate
lookup_same = {frozenset((a, b)): r for a, b, r in zip(same["A"], same["B"], same.to_dict("records"))}
added = []
for a, b in ADDED_PAIRS:
    key = frozenset((short[a], short[b]))
    if key not in lookup_same:
        raise ValueError(f"{a}-{b}: non sono nello stesso gruppo industriale o non nell'universo filtrato")
    added.append({**lookup_same[key], "origine": "aggiunta per scelta"})
final = pd.concat([final, pd.DataFrame(added)], ignore_index=True)
final.insert(0, "pair", final["A"].str.split().str[0] + "-" + final["B"].str.split().str[0])
dist_funnel["coppie aggiunte per scelta"] = len(added)
final[["pair", "origine", "industry", "rank_ssd", "ssd", "corr"]].round(3)

,pair,origine,industry,rank_ssd,ssd,corr
0,MA-V,regola,Servizi finanziari,1,3.854,0.880
1,TFC-USB,regola,Banche,3,4.717,0.830
2,BAC-PNC,regola,Banche,4,6.140,0.790
3,CSX-UNP,regola,Trasporto,5,6.610,0.717
4,PYPL-XYZ,regola,Servizi finanziari,6,8.737,0.612
5,BNY-GS,regola,Servizi finanziari,7,9.817,0.670
6,HD-LOW,regola,Beni discrezionali Distribuzio,8,10.391,0.868
7,ITW-OTIS,regola,Beni strumentali,10,10.630,0.615
8,MCD-YUM,regola,Servizi al consumatore,11,10.822,0.610
9,KO-PEP,aggiunta per scelta,"Alimenti, bevande e tabacco",240,55.347,0.682


## 6. Diagnostiche delle coppie scelte

Qui **non si scartano coppie**: la regola di selezione è già fissata. Le diagnostiche servono a conoscere meglio le coppie e a segnalare quelle da guardare con più attenzione nel report. Ogni misura ha il suo significato:

- **Ampiezza dello spread (%)**: deviazione standard del logaritmo del rapporto tra i due prezzi. Se è piccola rispetto ai costi (4 operazioni per trade) la strategia non può guadagnare. Valori di qualche punto percentuale sono ampi rispetto ai costi in basis points, ma su barre intraday l'ampiezza utile è molto più piccola: va rimisurata lì.
- **p-value di cointegrazione** (Engle-Granger sui log-prezzi, in entrambi i sensi, si tiene il peggiore): vicino a zero = lo spread tende a tornare verso la media. Non è un filtro perché sul giornaliero e su 5 anni è molto severo: anche le coppie "da manuale" faticano a superarlo (parte 8), e l'orizzonte del test (mesi) non è quello dei nostri trade (ore). Le stime vanno comunque riportate.
- **Hedge ratio β**: regressione del log-prezzo di A su quello di B. Con 1:1 (la nostra scelta di capitale uguale) β vicino a 1 significa che l'ipotesi è coerente con i dati.
- **Half-life** (tempo di dimezzamento dello scostamento, modello di Ornstein-Uhlenbeck: Chan, 2013): sul giornaliero è in giorni; i nostri trade durano ore, quindi è solo un'indicazione di quanto "elastico" è il legame. Va ricalcolata sulle barre intraday.
- **Attraversamenti della media all'anno**: quante volte lo spread torna alla sua media. Pochi attraversamenti = pochi trade.
- **SSD nelle due metà del periodo**: se la coppia era vicina solo in una metà, il legame non è stabile.

In [49]:
logp = np.log(prices)
half_a, half_b = prices.index[: len(prices) // 2], prices.index[len(prices) // 2 :]

def half_life(spread):
    lam = np.polyfit(spread[:-1], np.diff(spread), 1)[0]
    return -np.log(2) / lam if lam < 0 else np.inf

def crossings(spread):
    s = spread - spread.mean()
    return int((np.sign(s[1:]) != np.sign(s[:-1])).sum())

def worst_coint_p(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return max(coint(u, v)[1], coint(v, u)[1])

def ssd_on(a, b, idx):
    d = prices.loc[idx, a] / prices.loc[idx, a].iloc[0] - prices.loc[idx, b] / prices.loc[idx, b].iloc[0]
    return float((d ** 2).sum())

def beta_on(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return np.polyfit(v, u, 1)[0]

def diagnostics(a, b):
    lr = np.log(prices[a] / prices[b]).values
    beta = beta_on(a, b)
    p1, p2, b1, b2 = worst_coint_p(a, b, half_a), worst_coint_p(a, b, half_b), beta_on(a, b, half_a), beta_on(a, b, half_b)
    return dict(
        spread_std_pct=lr.std() * 100, coint_p=worst_coint_p(a, b), beta=beta, half_life=half_life(lr),
        crossings_per_year=crossings(lr) / years, ssd_first=ssd_on(a, b, half_a), ssd_second=ssd_on(a, b, half_b),
        ok_coint=worst_coint_p(a, b) < COINT_P, ok_beta=BETA_RANGE[0] <= beta <= BETA_RANGE[1],
        ok_hl=HL_RANGE[0] <= half_life(lr) <= HL_RANGE[1], ok_cross=crossings(lr) / years >= MIN_CROSSINGS_PER_YEAR,
        ok_stable=(p1 < STAB_P) and (p2 < STAB_P) and abs(b1 - b2) / abs(beta) <= STAB_BETA_TOL,
    )

diag = pd.DataFrame([diagnostics(a, b) for a, b in zip(final["A"], final["B"])])
final_d = pd.concat([final[["pair", "origine", "A", "B", "industry", "rank_ssd", "ssd", "corr"]], diag], axis=1)
checks = ["ok_coint", "ok_beta", "ok_hl", "ok_cross", "ok_stable"]
final_d["da_guardare"] = final_d[checks].apply(lambda r: ", ".join(c[3:] for c in checks if not r[c]) or "-", axis=1)
final_d.drop(columns=["A", "B"]).round(3)

,pair,origine,industry,rank_ssd,ssd,corr,spread_std_pct,coint_p,beta,half_life,crossings_per_year,ssd_first,ssd_second,ok_coint,ok_beta,ok_hl,ok_cross,ok_stable,da_guardare
0,MA-V,regola,Servizi finanziari,1,3.854,0.880,3.336,0.081,0.985,32.121,14.457,0.988,1.380,False,True,False,True,False,"coint, hl, stable"
1,TFC-USB,regola,Banche,3,4.717,0.830,6.621,0.370,1.070,49.130,12.249,2.057,10.672,False,True,False,True,False,"coint, hl, stable"
2,BAC-PNC,regola,Banche,4,6.140,0.790,5.905,0.066,1.049,39.792,17.068,0.812,13.603,False,True,False,True,False,"coint, hl, stable"
3,CSX-UNP,regola,Trasporto,5,6.610,0.717,6.339,0.505,0.950,56.479,8.433,1.342,5.246,False,True,False,False,False,"coint, hl, cross, stable"
4,PYPL-XYZ,regola,Servizi finanziari,6,8.737,0.612,17.732,0.557,0.985,62.121,13.453,4.941,25.585,False,True,False,True,False,"coint, hl, stable"
5,BNY-GS,regola,Servizi finanziari,7,9.817,0.670,6.380,0.037,1.009,30.560,12.048,3.629,6.061,True,True,False,True,False,"hl, stable"
6,HD-LOW,regola,Beni discrezionali Distribuzio,8,10.391,0.868,5.152,0.144,0.975,52.404,7.630,4.818,3.785,False,True,False,False,False,"coint, hl, cross, stable"
7,ITW-OTIS,regola,Beni strumentali,10,10.630,0.615,7.949,0.923,0.805,109.724,11.847,1.450,6.846,False,True,False,False,False,"coint, hl, cross, stable"
8,MCD-YUM,regola,Servizi al consumatore,11,10.822,0.610,5.819,0.122,0.892,44.139,10.843,4.618,9.287,False,True,False,False,False,"coint, hl, cross, stable"
9,KO-PEP,aggiunta per scelta,"Alimenti, bevande e tabacco",240,55.347,0.682,15.364,0.946,0.052,inf,2.209,4.932,47.818,False,False,False,False,False,"coint, beta, hl, cross, stable"


**Come leggere la tabella.** La colonna `da_guardare` elenca i controlli non superati (coint = cointegrazione, beta, hl = half-life, cross = attraversamenti, stable = stabilità nelle due metà). Una coppia con molti controlli falliti non è esclusa, ma nel report va descritta come più rischiosa, e sarà la fase intraday a dire se funziona.

## 7. Canale di confronto: cointegrazione sulle coppie dello stesso settore

Lo stesso universo (stesso periodo, stessi titoli, stesso settore) viene passato nell'imbuto della cointegrazione, il metodo alternativo (Engle e Granger, 1987; Vidyamurthy, 2004). Serve a un solo scopo: confrontare i due metodi e vedere se la distanza ha scelto coppie che la cointegrazione boccerebbe del tutto.

Poiché si provano molte coppie, si applica la correzione FDR (Benjamini-Hochberg): senza di essa circa il 5% delle coppie risulterebbe cointegrato per puro caso con p < 0.05.

In [50]:
cand_c = same[(same["corr"] >= MIN_CORR_COINT) & (same["corr"] <= MAX_CORR_SAME_COMPANY)]
coint_res = pd.DataFrame([{**dict(A=a, B=b), **diagnostics(a, b)} for a, b in zip(cand_c["A"], cand_c["B"])])
coint_res["corr"] = cand_c["corr"].values
coint_res["p_fdr"] = multipletests(coint_res["coint_p"], method="fdr_bh")[1]
coint_res["ok_fdr"] = coint_res["p_fdr"] < FDR_ALPHA

c_funnel = {"stessa industria": len(same), f"corr tra {MIN_CORR_COINT} e {MAX_CORR_SAME_COMPANY}": len(coint_res)}
r = coint_res[coint_res["ok_coint"]];  c_funnel[f"cointegrazione p < {COINT_P}"] = len(r)
r = r[r["ok_beta"]];                   c_funnel[f"hedge ratio {BETA_RANGE}"] = len(r)
r = r[r["ok_hl"]];                     c_funnel[f"half-life {HL_RANGE} giorni"] = len(r)
r = r[r["ok_cross"]];                  c_funnel[f"attraversamenti >= {MIN_CROSSINGS_PER_YEAR}/anno"] = len(r)
r = r[r["ok_stable"]];                 c_funnel["stabili nelle due meta"] = len(r)
r = r[r["ok_fdr"]];                    c_funnel["dopo FDR"] = len(r)
pd.Series(c_funnel, name="n").to_frame()

,n
stessa industria,2390
corr tra 0.7 e 0.97,173
cointegrazione p < 0.05,11
"hedge ratio (0.5, 2.0)",9
"half-life (1, 30) giorni",1
attraversamenti >= 12/anno,1
stabili nelle due meta,0
dopo FDR,0


**Lettura.** Se l'imbuto lascia pochissime coppie è un risultato, non un errore: la cointegrazione sul giornaliero su 5 anni è un criterio severo e "lento". Non lo allentiamo per far uscire più coppie (sarebbe snooping): lo riportiamo così com'è. Le coppie che passano *entrambi* i canali sono le più solide.

In [51]:
final_d["passa_cointegrazione"] = [frozenset((a, b)) in {frozenset((x, y)) for x, y in zip(r["A"], r["B"])} for a, b in zip(final["A"], final["B"])]
print("Coppie scelte dalla distanza che passano anche l'imbuto della cointegrazione:", int(final_d["passa_cointegrazione"].sum()), "su", len(final_d))
r[["A", "B", "corr", "coint_p", "p_fdr", "beta", "half_life", "crossings_per_year"]].round(3)

Coppie scelte dalla distanza che passano anche l'imbuto della cointegrazione: 0 su 10


,A,B,corr,coint_p,p_fdr,beta,half_life,crossings_per_year


## 8. Controllo: le coppie "da manuale" (candidate teoriche)

Elenco deciso **prima** di guardare i dati, 14 coppie: sono esempi ricorrenti in manuali, tutorial e progetti universitari, non coppie validate da un articolo specifico. Servono come test di buon senso: se la regola le piazza in alto, il metodo si comporta come ci aspettiamo; se non le trova, lo scriviamo (non le forziamo).

**Attenzione alle fonti.** I riferimenti di base sono Gatev et al. (2006) e Vidyamurthy (2004). KO-PEP in particolare è citata come esempio in alcuni lavori ma risulta non cointegrata in altri, a seconda del periodo.

In [52]:
THEORETICAL = [
    ("KO", "PEP"), ("V", "MA"), ("HD", "LOW"), ("XOM", "CVX"), ("BAC", "WFC"), ("GS", "MS"), ("UPS", "FDX"),
    ("LMT", "NOC"), ("PG", "CL"), ("MCD", "YUM"), ("DUK", "SO"), ("MRK", "PFE"), ("T", "VZ"), ("CME", "ICE"),
]
lookup = {frozenset((a, b)): row for a, b, row in zip(same["A"], same["B"], same.to_dict("records"))}
sel = {frozenset((a, b)) for a, b, o in zip(final["A"], final["B"], final["origine"]) if o == "regola"}
added_keys = {frozenset((short[a], short[b])) for a, b in ADDED_PAIRS}
rows = []
for a, b in THEORETICAL:
    ta, tb = short.get(a), short.get(b)
    key = frozenset((ta, tb))
    if ta not in names or tb not in names:
        rows.append(dict(pair=f"{a}-{b}", esito="non nell'universo filtrato (liquidita o dati)")); continue
    if key not in lookup:
        rows.append(dict(pair=f"{a}-{b}", esito="industria GICS diversa nel file Bloomberg", **{"industria A": gics.get(ta), "industria B": gics.get(tb)})); continue
    row = lookup[key]
    d = diagnostics(ta, tb)
    esito = "SELEZIONATA" if key in sel else "AGGIUNTA per scelta" if key in added_keys else ("scartata: quasi identiche" if row["corr"] > MAX_CORR_SAME_COMPANY
             else ("scartata: correlazione bassa" if row["corr"] < MIN_CORR else "non nelle prime per SSD"))
    rows.append(dict(pair=f"{a}-{b}", esito=esito, rank_ssd=row["rank_ssd"], su=len(same), corr=row["corr"],
                     coint_p=d["coint_p"], half_life=d["half_life"], crossings_per_year=d["crossings_per_year"]))
pd.DataFrame(rows).round(3)

,pair,esito,rank_ssd,su,corr,coint_p,half_life,crossings_per_year
0,KO-PEP,AGGIUNTA per scelta,240,2390,0.682,0.946,inf,2.209
1,V-MA,SELEZIONATA,1,2390,0.880,0.081,32.121,14.457
2,HD-LOW,SELEZIONATA,8,2390,0.868,0.144,52.404,7.630
3,XOM-CVX,non nelle prime per SSD,744,2390,0.852,0.415,257.529,5.020
4,BAC-WFC,non nelle prime per SSD,866,2390,0.820,0.491,156.639,4.217
5,GS-MS,non nelle prime per SSD,204,2390,0.841,0.048,157.460,3.815
6,UPS-FDX,non nelle prime per SSD,578,2390,0.601,0.645,inf,1.807
7,LMT-NOC,non nelle prime per SSD,65,2390,0.640,0.244,57.718,9.638
8,PG-CL,non nelle prime per SSD,118,2390,0.720,0.065,62.264,8.835
9,MCD-YUM,SELEZIONATA,11,2390,0.610,0.122,44.139,10.843


## 8b. Robustezza al periodo di selezione

**Domanda:** le coppie scelte sui 5 anni restano vicine anche usando più storia (10, 15, 20 anni)? La stessa regola (stessi filtri, stessa distanza, correlazione ≥ `MIN_CORR`, un titolo per coppia, tetto per industria) viene rifatta con inizio diverso e la stessa fine, il 15 giugno 2026.

**Cosa non è:** non serve a scegliere il periodo né a cambiare la lista. Scegliere il periodo "che dà le coppie più belle" sarebbe adattare la scelta ai dati. La lista resta quella dei 5 anni (più KO-PEP aggiunta per scelta); qui si misura quanto è stabile.

**Attenzione a leggere i risultati:**
- Più anni significano meno titoli con storia completa (il survivorship bias cresce: restano le aziende che esistevano già) e più regimi di mercato (2008, 2020) che muovono tutto insieme.
- Un titolo non presente nell'universo di una finestra (n.d.) non ha il suo posto in classifica: non vuol dire che la coppia sia peggiore, solo che quel periodo non è confrontabile.

In [53]:
daily_full = load("daily")
daily_full["ticker"] = daily_full["ticker"].astype(str)

def select_pairs(start):
    """Stessa regola della parte 5, con un altro inizio del periodo."""
    d = daily_full[(daily_full["date"] >= start) & (daily_full["date"] <= SEL_END)]
    px = d.pivot(index="date", columns="ticker", values="PX_LAST")
    vol = d.pivot(index="date", columns="ticker", values="PX_VOLUME")
    ret = d.pivot(index="date", columns="ticker", values="ret")
    ok = ((px * vol).median() >= MIN_DOLLAR_VOLUME) & (px.isna().mean() <= MAX_MISSING_PCT)
    px, ret = px.loc[:, ok].ffill(), ret.loc[:, ok]
    comp = px.columns[px.notna().all()]
    px, ret = px[comp], ret[comp]
    nm = list(px.columns)
    x = (px / px.iloc[0]).values.T
    sq = (x ** 2).sum(axis=1)
    ssd = sq[:, None] + sq[None, :] - 2 * x @ x.T
    iu_ = np.triu_indices(len(nm), k=1)
    ind_ = gics.reindex(nm).values
    c_ = ret.corr().reindex(index=nm, columns=nm).values
    df = pd.DataFrame({"A": [nm[i] for i in iu_[0]], "B": [nm[j] for j in iu_[1]], "ssd": ssd[iu_], "corr": c_[iu_],
                       "industry": ind_[iu_[0]], "same": ind_[iu_[0]] == ind_[iu_[1]]})
    df = df[df["same"]].sort_values("ssd").reset_index(drop=True)
    df["rank_ssd"] = np.arange(1, len(df) + 1)
    cand_ = df[(df["corr"] <= MAX_CORR_SAME_COMPANY) & (df["corr"] >= MIN_CORR)]
    chosen_, used_, per_ = [], set(), {}
    for _, r_ in cand_.iterrows():
        if r_["A"] in used_ or r_["B"] in used_ or per_.get(r_["industry"], 0) >= MAX_PER_INDUSTRY:
            continue
        chosen_.append(frozenset((r_["A"], r_["B"]))); used_.update([r_["A"], r_["B"]]); per_[r_["industry"]] = per_.get(r_["industry"], 0) + 1
        if len(chosen_) == N_PAIRS_FINAL:
            break
    return dict(n_stocks=len(nm), n_same=len(df), ranks={frozenset((a, b)): k for a, b, k in zip(df["A"], df["B"], df["rank_ssd"])},
                chosen=chosen_, universe=set(nm))

results = {label: select_pairs(start) for label, start in ROBUSTNESS_STARTS.items()}

# controllo: la finestra di 5 anni deve riprodurre le coppie scelte dalla regola
first_label = next(iter(results))
rule_keys = {frozenset((a, b)) for a, b, o in zip(final["A"], final["B"], final["origine"]) if o == "regola"}
print(f"Finestra {first_label} uguale alla selezione della parte 5:", set(results[first_label]["chosen"]) == rule_keys)

summary = pd.DataFrame({label: dict(titoli_con_dati_completi=r["n_stocks"], coppie_stessa_industria=r["n_same"]) for label, r in results.items()}).T
summary

Finestra 5 anni uguale alla selezione della parte 5: True


,titoli_con_dati_completi,coppie_stessa_industria
5 anni,307,2390
10 anni,226,1267
15 anni,143,512
20 anni,104,300


In [54]:
# per ogni coppia della lista finale: posto per SSD in ciascuna finestra, con una spunta se e tra le 9 scelte dalla regola
table_rows = []
for _, row in final_d.iterrows():
    key = frozenset((row["A"], row["B"]))
    out_row = {"pair": row["pair"], "origine": row["origine"]}
    n_sel = 0
    for label, r_ in results.items():
        if row["A"] not in r_["universe"] or row["B"] not in r_["universe"]:
            out_row[label] = "n.d."
        else:
            sel_ = key in r_["chosen"]
            n_sel += sel_
            out_row[label] = f"{r_['ranks'][key]}" + (" ✓" if sel_ else "")
    out_row["finestre_in_cui_e_scelta"] = n_sel
    table_rows.append(out_row)
robust = pd.DataFrame(table_rows)
final_d["finestre_selezionata"] = robust["finestre_in_cui_e_scelta"].values
robust

,pair,origine,5 anni,10 anni,15 anni,20 anni,finestre_in_cui_e_scelta
0,MA-V,regola,1 ✓,417,107,n.d.,1
1,TFC-USB,regola,3 ✓,8 ✓,n.d.,n.d.,2
2,BAC-PNC,regola,4 ✓,283,8 ✓,77,2
3,CSX-UNP,regola,5 ✓,407,n.d.,n.d.,1
4,PYPL-XYZ,regola,6 ✓,1190,n.d.,n.d.,1
5,BNY-GS,regola,7 ✓,620,n.d.,n.d.,1
6,HD-LOW,regola,8 ✓,13,69,112,1
7,ITW-OTIS,regola,10 ✓,n.d.,n.d.,n.d.,1
8,MCD-YUM,regola,11 ✓,6 ✓,n.d.,n.d.,2
9,KO-PEP,aggiunta per scelta,240,n.d.,n.d.,n.d.,0


In [55]:
# coppie scelte dalla regola in almeno 2 finestre, anche se non sono nella nostra lista dei 5 anni
from collections import Counter
count = Counter(k for r_ in results.values() for k in r_["chosen"])
persist = pd.DataFrame([{"pair": "-".join(t.split()[0] for t in sorted(k)), "finestre": n,
                         "in_lista_finale": k in {frozenset((a, b)) for a, b in zip(final["A"], final["B"])}}
                        for k, n in count.items() if n >= 2]).sort_values(["finestre", "pair"], ascending=[False, True]).reset_index(drop=True)
persist

,pair,finestre,in_lista_finale
0,HAL-SLB,3,False
1,BAC-PNC,2,True
2,CB-TRV,2,False
3,COP-CVX,2,False
4,MCD-YUM,2,True
5,T-VZ,2,False
6,TFC-USB,2,True
7,USB-WFC,2,False


**Come leggere le tabelle.** Nella prima tabella, ogni cella è il posto della coppia per SSD tra tutte le coppie dello stesso gruppo industriale in quel periodo (1 = la più vicina); ✓ vuol dire che è tra le 9 scelte dalla regola in quella finestra; n.d. che uno dei due titoli non ha dati completi in quel periodo. La seconda tabella mostra le coppie scelte dalla regola in almeno due finestre.

**Cosa ci dice, e cosa no.** Una coppia che compare in più finestre ha un legame meno dipendente da un solo periodo: è un segnale di robustezza, da riportare nel report. Una coppia che compare solo nei 5 anni non è per questo scartata: la lista è fissata a priori. Le coppie che emergono solo nei periodi più lunghi (spesso quelle "da manuale", come HAL-SLB) sono un'informazione per il report e per sviluppi futuri, **non** una modifica della lista: cambiarla ora, dopo aver visto che i periodi lunghi danno coppie economicamente più pulite, sarebbe adattare la scelta ai risultati.

## 9. Riepilogo per il report

**Imbuto del metodo della distanza** (da riportare con i numeri a ogni passaggio):

In [56]:
pd.Series(dist_funnel, name="n").to_frame()

,n
coppie possibili (tutti i titoli),46971
stessa industria,2390
escluse quasi identiche (corr > 0.97),2389
correlazione >= 0.6,467
coppie finali (dalla regola),9
coppie aggiunte per scelta,1


**Limiti da dichiarare nel report**
1. **Survivorship bias**: l'universo sono i titoli dell'S&P 500 di oggi.
2. **GICS attuale**, non storico (riclassificazione del 2018).
3. **Un solo periodo di selezione** di 5 anni: le coppie sono scelte sul passato e il legame può cambiare. Il solo test vero è l'intraday, usato una volta sola.
4. **Soglie e livello GICS nostri** (liquidità, correlazione minima 0.6, tetto per industria, un titolo per coppia, `industry_group` invece di `sub_industry`) senza fonte accademica: sono scelte ragionevoli, dichiarate prima dei risultati, ma non ottimizzate.
5. **Le diagnostiche giornaliere** (half-life, cointegrazione) misurano orizzonti di giorni e mesi, mentre i trade durano ore: vanno ricalcolate sull'intraday.
6. **Test multipli**: la distanza classifica migliaia di coppie, quindi ci sono comunque coppie "vicine per caso". Il numero di coppie testate va riportato.
7. **Ordine delle scelte**: alcune regole (correlazione minima, tetto per industria) sono state definite dopo aver visto i primi risultati della selezione sul periodo giornaliero. Il test intraday resta pulito perché non è stato usato per scegliere nulla.
8. **KO-PEP aggiunta per scelta** (decima coppia, fuori dalla regola e oltre le 6-9 del programma): ha senso economico ma nel periodo scelto non è tra le coppie più vicine e non risulta cointegrata. Va presentata e valutata a parte (risultati con e senza), non come coppia selezionata dai dati.
9. **Dipendenza dal periodo**: ripetendo la selezione su 10, 15 e 20 anni le coppie cambiano molto (e con 20 anni solo circa un quinto dei titoli ha storia completa: più survivorship bias). La lista dei 5 anni resta quella scelta a priori; la parte 8b riporta quali coppie restano stabili.

**Riferimenti** (da verificare prima di citarli: non ho potuto leggere i testi completi)
- Benjamini, Y., Hochberg, Y. (1995), *Controlling the false discovery rate*, Journal of the Royal Statistical Society B 57(1).
- Chan, E. (2013), *Algorithmic Trading: Winning Strategies and Their Rationale*, Wiley.
- Engle, R., Granger, C. (1987), *Co-integration and error correction*, Econometrica 55(2).
- Gatev, E., Goetzmann, W., Rouwenhorst, K. G. (2006), *Pairs Trading: Performance of a Relative-Value Arbitrage Rule*, Review of Financial Studies 19(3), 797-827.
- Harvey, C., Liu, Y., Zhu, H. (2016), *...and the Cross-Section of Expected Returns*, Review of Financial Studies 29(1).
- Krauss, C. (2017), *Statistical arbitrage pairs trading strategies: Review and outlook*, Journal of Economic Surveys 31(2).
- Lo, A., MacKinlay, A. C. (1990), *Data-snooping biases in tests of financial asset pricing models*, Review of Financial Studies 3(3).
- Vidyamurthy, G. (2004), *Pairs Trading: Quantitative Methods and Analysis*, Wiley.

In [57]:
# salva la lista finale: serve alla fase intraday
out = final_d[["pair", "origine", "A", "B", "industry", "rank_ssd", "ssd", "corr", "da_guardare"]].copy()
out.to_csv(Path("../data/processed/selected_pairs.csv"), index=False)
out

,pair,origine,A,B,industry,rank_ssd,ssd,corr,da_guardare
0,MA-V,regola,MA UN Equity,V UN Equity,Servizi finanziari,1,3.853901,0.880413,"coint, hl, stable"
1,TFC-USB,regola,TFC UN Equity,USB UN Equity,Banche,3,4.716768,0.830456,"coint, hl, stable"
2,BAC-PNC,regola,BAC UN Equity,PNC UN Equity,Banche,4,6.139650,0.789725,"coint, hl, stable"
3,CSX-UNP,regola,CSX UW Equity,UNP UN Equity,Trasporto,5,6.609540,0.716640,"coint, hl, cross, stable"
4,PYPL-XYZ,regola,PYPL UW Equity,XYZ UN Equity,Servizi finanziari,6,8.736951,0.611904,"coint, hl, stable"
5,BNY-GS,regola,BNY UN Equity,GS UN Equity,Servizi finanziari,7,9.816665,0.669978,"hl, stable"
6,HD-LOW,regola,HD UN Equity,LOW UN Equity,Beni discrezionali Distribuzio,8,10.391419,0.867847,"coint, hl, cross, stable"
7,ITW-OTIS,regola,ITW UN Equity,OTIS UN Equity,Beni strumentali,10,10.630223,0.614919,"coint, hl, cross, stable"
8,MCD-YUM,regola,MCD UN Equity,YUM UN Equity,Servizi al consumatore,11,10.821660,0.610235,"coint, hl, cross, stable"
9,KO-PEP,aggiunta per scelta,KO UN Equity,PEP UW Equity,"Alimenti, bevande e tabacco",240,55.347367,0.681910,"coint, beta, hl, cross, stable"


## 10. Risultati e coppie scelte (testo per il report)

La cella seguente genera il testo dai risultati del notebook: se si rilancia con altri dati o parametri, numeri e tabella si aggiornano da soli. Le descrizioni economiche delle singole coppie sono scritte a mano nel dizionario `ECON` e vanno riviste se la lista cambia.

In [58]:
from IPython.display import Markdown, display

# descrizione economica breve di ogni coppia (testo scritto a mano; per coppie nuove resta il testo generico)
ECON = {
    "KO-PEP": "i due colossi delle bevande analcoliche: stesso mercato, stessi clienti e canali di vendita. ECCEZIONE: aggiunta per scelta del gruppo, nei dati del periodo non è tra le coppie più vicine (vedi tabella) e non risulta cointegrata.",
    "MA-V": "le due grandi reti di pagamento con carta: stesso modello di business, stessi clienti, ricavi legati ai volumi di spesa dei consumatori.",
    "TFC-USB": "banche commerciali USA di dimensione simile: stessa esposizione ai tassi d'interesse e al credito retail.",
    "BAC-PNC": "banche commerciali USA (BAC più grande, PNC regionale): stessi fattori di tassi e credito, ma business mix non identico.",
    "CSX-UNP": "due ferrovie merci USA: stessi volumi di carico, stessa regolamentazione e stessi costi (carburante, lavoro).",
    "PYPL-XYZ": "pagamenti digitali per consumatori e commercianti (XYZ è Block): settore comune ma ampiezza dello spread molto alta e correlazione bassa, coppia più rischiosa.",
    "BNY-GS": "servizi finanziari: BNY è custodia e gestione, GS è investment banking; legame economico più debole.",
    "HD-LOW": "i due grandi distributori USA per la casa e il fai-da-te: stessi clienti, stessa stagionalità, stessa sensibilità al mercato immobiliare.",
    "ITW-OTIS": "beni strumentali (componenti industriali contro ascensori): stesso gruppo GICS ma business diversi, legame debole e half-life lunga.",
    "MCD-YUM": "ristorazione rapida in franchising: stessa struttura dei ricavi e sensibilità alla spesa dei consumatori.",
}
f = list(dist_funnel.values())
n_pairs_all, n_same, n_noident, n_corr, n_rule = f[:5]
n_final = len(final_d)
n_added = n_final - n_rule
rule_d = final_d[final_d["origine"] == "regola"]
n_stocks = len(names)
it = lambda n: f"{n:,}".replace(",", ".")     # 46971 -> 46.971
n_flags = final_d["da_guardare"].apply(lambda t: 0 if t == "-" else len(t.split(",")))
risky = ", ".join(final_d.loc[n_flags == n_flags.max(), "pair"])
others = sorted((r_ for r_ in rows if r_.get("esito") == "non nelle prime per SSD"), key=lambda r_: r_["rank_ssd"])[:3]
kopep = "; ".join(f"{r_['pair']} al posto {int(r_['rank_ssd'])} su {it(n_same)}" for r_ in others) or "nessun dettaglio"
added_names = ", ".join(final_d.loc[final_d["origine"] != "regola", "pair"])
sel_theory = [r["pair"] for r in rows if r.get("esito") == "SELEZIONATA"]

table = "| Coppia | Origine | Industria GICS | Posto SSD (su %d) | Correlazione | Ampiezza spread | Da guardare |\n|---|---|---|---|---|---|---|\n" % n_same
for _, r_ in final_d.iterrows():
    table += f"| **{r_['pair']}** | {r_['origine']} | {r_['industry']} | {int(r_['rank_ssd'])} | {r_['corr']:.2f} | {r_['spread_std_pct']:.1f}% | {r_['da_guardare']} |\n"

motivations = ""
for _, r_ in final_d.iterrows():
    motivations += f"- **{r_['pair']}**: {ECON.get(r_['pair'], 'stesso gruppo industriale GICS.')}\n"

text = f"""
## 10. Risultati e coppie scelte

### Come abbiamo scelto
Abbiamo usato il **metodo della distanza** (Gatev, Goetzmann e Rouwenhorst, 2006) sui prezzi giornalieri aggiustati dal {SEL_START} al {SEL_END}, un periodo che finisce prima dell'inizio dei dati intraday per non vedere il periodo di test. Partiamo da {n_stocks} titoli liquidi (valore scambiato mediano sopra {MIN_DOLLAR_VOLUME/1e6:.0f} milioni di dollari al giorno) con dati completi. Le coppie possibili sono {it(n_pairs_all)}, ma ne consideriamo solo {it(n_same)}: quelle dello stesso gruppo industriale GICS (`{GICS_LEVEL}`), perché aziende con lo stesso business sono esposte agli stessi fattori economici. Togliamo le coppie quasi identiche (correlazione dei rendimenti sopra {MAX_CORR_SAME_COMPANY}, per esempio due classi di azioni della stessa società, il cui spread è troppo piccolo per coprire i costi) e quelle con correlazione sotto {MIN_CORR}, che non si muovono abbastanza insieme: restano {n_corr}. Le ordiniamo per SSD crescente (più vicine nel passato) e prendiamo le prime {N_PAIRS_FINAL}, con un solo titolo per coppia e al massimo {MAX_PER_INDUSTRY} coppie per industria.

A queste aggiungiamo per decisione del gruppo {n_added} coppia ({added_names}), **fuori dalla regola**: ha una chiara giustificazione economica (stesso mercato, stessi clienti), ma nei dati non è tra le più vicine, quindi nel report va presentata come scelta nostra e valutata a parte (risultati con e senza).

### Le {n_final} coppie ({n_rule} dalla regola, {n_added} aggiunta per scelta)
{table}
Il posto in classifica è quello tra tutte le {it(n_same)} coppie dello stesso gruppo: i posti mancanti sono coppie scartate dalle regole (quasi identiche, correlazione bassa, titolo già usato in un'altra coppia).

**Perché queste coppie (motivazione economica):**
{motivations}
### Cosa abbiamo trovato
- Le {n_rule} coppie selezionate dalla regola sono quelle dello stesso settore che si sono mosse più vicine nel periodo, con correlazione dei rendimenti tra {rule_d['corr'].min():.2f} e {rule_d['corr'].max():.2f}; la coppia aggiunta per scelta ({added_names}) ha invece una correlazione di {final_d.loc[final_d['origine'] != 'regola', 'corr'].iloc[0]:.2f} e un posto in classifica molto più basso.
- **Test di cointegrazione** (canale di confronto): tra le {n_final} coppie scelte, quelle con p-value sotto {COINT_P} sono {int(final_d['ok_coint'].sum())}, e quelle che passano l'intero imbuto della cointegrazione (con stabilità e correzione FDR) sono {int(final_d['passa_cointegrazione'].sum())}. Il legame statistico sul giornaliero e su 5 anni è quindi debole: i prezzi si muovono vicini, ma non c'è una forte prova di ritorno alla media.
- **Robustezza al periodo**: ripetendo la stessa regola su 10, 15 e 20 anni, {int((final_d['finestre_selezionata'] >= 2).sum())} delle {n_final} coppie della lista compaiono tra le 9 scelte in almeno 2 delle {len(results)} finestre ({', '.join(final_d.loc[final_d['finestre_selezionata'] >= 2, 'pair']) or 'nessuna'}). Le coppie più "pulite" dei periodi lunghi (per esempio le classiche dell'energia) non sono state aggiunte alla lista: sono un'informazione per il report, non una modifica della selezione a posteriori.
- **Coppie "da manuale"**: di quelle che avevamo indicato in anticipo, la regola ne seleziona {len(sel_theory)} ({', '.join(sel_theory) if sel_theory else 'nessuna'}), e ne abbiamo aggiunta una per scelta ({added_names}). Le altre si piazzano più in basso nella classifica per SSD o hanno correlazione troppo bassa: {kopep}.

### Cosa ci aspettiamo e cosa verificheremo
- La selezione è un **elenco di candidate**: la prova che le coppie funzionano arriva solo dall'addestramento e dal test sui dati intraday, usato una sola volta.
- Half-life e attraversamenti sul giornaliero misurano giorni, mentre i nostri trade durano ore: vanno ricalcolati sull'intraday.
- Le coppie con più segnalazioni nella colonna "Da guardare" ({risky}) sono quelle con più rischi; PYPL-XYZ, per l'ampiezza dello spread e la correlazione bassa, e ITW-OTIS, per l'half-life lunga, vanno guardate con particolare attenzione. Nel report vanno descritte con i loro rischi.
- I limiti elencati sopra (survivorship bias, GICS attuale, soglie nostre, test multipli, scelte fissate dopo aver visto i primi risultati giornalieri) vanno dichiarati nel report.
"""
display(Markdown(text))



## 10. Risultati e coppie scelte

### Come abbiamo scelto
Abbiamo usato il **metodo della distanza** (Gatev, Goetzmann e Rouwenhorst, 2006) sui prezzi giornalieri aggiustati dal 2021-06-16 al 2026-06-15, un periodo che finisce prima dell'inizio dei dati intraday per non vedere il periodo di test. Partiamo da 307 titoli liquidi (valore scambiato mediano sopra 50 milioni di dollari al giorno) con dati completi. Le coppie possibili sono 46.971, ma ne consideriamo solo 2.390: quelle dello stesso gruppo industriale GICS (`industry_group`), perché aziende con lo stesso business sono esposte agli stessi fattori economici. Togliamo le coppie quasi identiche (correlazione dei rendimenti sopra 0.97, per esempio due classi di azioni della stessa società, il cui spread è troppo piccolo per coprire i costi) e quelle con correlazione sotto 0.6, che non si muovono abbastanza insieme: restano 467. Le ordiniamo per SSD crescente (più vicine nel passato) e prendiamo le prime 9, con un solo titolo per coppia e al massimo 3 coppie per industria.

A queste aggiungiamo per decisione del gruppo 1 coppia (KO-PEP), **fuori dalla regola**: ha una chiara giustificazione economica (stesso mercato, stessi clienti), ma nei dati non è tra le più vicine, quindi nel report va presentata come scelta nostra e valutata a parte (risultati con e senza).

### Le 10 coppie (9 dalla regola, 1 aggiunta per scelta)
| Coppia | Origine | Industria GICS | Posto SSD (su 2390) | Correlazione | Ampiezza spread | Da guardare |
|---|---|---|---|---|---|---|
| **MA-V** | regola | Servizi finanziari | 1 | 0.88 | 3.3% | coint, hl, stable |
| **TFC-USB** | regola | Banche | 3 | 0.83 | 6.6% | coint, hl, stable |
| **BAC-PNC** | regola | Banche | 4 | 0.79 | 5.9% | coint, hl, stable |
| **CSX-UNP** | regola | Trasporto | 5 | 0.72 | 6.3% | coint, hl, cross, stable |
| **PYPL-XYZ** | regola | Servizi finanziari | 6 | 0.61 | 17.7% | coint, hl, stable |
| **BNY-GS** | regola | Servizi finanziari | 7 | 0.67 | 6.4% | hl, stable |
| **HD-LOW** | regola | Beni discrezionali Distribuzio | 8 | 0.87 | 5.2% | coint, hl, cross, stable |
| **ITW-OTIS** | regola | Beni strumentali | 10 | 0.61 | 7.9% | coint, hl, cross, stable |
| **MCD-YUM** | regola | Servizi al consumatore | 11 | 0.61 | 5.8% | coint, hl, cross, stable |
| **KO-PEP** | aggiunta per scelta | Alimenti, bevande e tabacco | 240 | 0.68 | 15.4% | coint, beta, hl, cross, stable |

Il posto in classifica è quello tra tutte le 2.390 coppie dello stesso gruppo: i posti mancanti sono coppie scartate dalle regole (quasi identiche, correlazione bassa, titolo già usato in un'altra coppia).

**Perché queste coppie (motivazione economica):**
- **MA-V**: le due grandi reti di pagamento con carta: stesso modello di business, stessi clienti, ricavi legati ai volumi di spesa dei consumatori.
- **TFC-USB**: banche commerciali USA di dimensione simile: stessa esposizione ai tassi d'interesse e al credito retail.
- **BAC-PNC**: banche commerciali USA (BAC più grande, PNC regionale): stessi fattori di tassi e credito, ma business mix non identico.
- **CSX-UNP**: due ferrovie merci USA: stessi volumi di carico, stessa regolamentazione e stessi costi (carburante, lavoro).
- **PYPL-XYZ**: pagamenti digitali per consumatori e commercianti (XYZ è Block): settore comune ma ampiezza dello spread molto alta e correlazione bassa, coppia più rischiosa.
- **BNY-GS**: servizi finanziari: BNY è custodia e gestione, GS è investment banking; legame economico più debole.
- **HD-LOW**: i due grandi distributori USA per la casa e il fai-da-te: stessi clienti, stessa stagionalità, stessa sensibilità al mercato immobiliare.
- **ITW-OTIS**: beni strumentali (componenti industriali contro ascensori): stesso gruppo GICS ma business diversi, legame debole e half-life lunga.
- **MCD-YUM**: ristorazione rapida in franchising: stessa struttura dei ricavi e sensibilità alla spesa dei consumatori.
- **KO-PEP**: i due colossi delle bevande analcoliche: stesso mercato, stessi clienti e canali di vendita. ECCEZIONE: aggiunta per scelta del gruppo, nei dati del periodo non è tra le coppie più vicine (vedi tabella) e non risulta cointegrata.

### Cosa abbiamo trovato
- Le 9 coppie selezionate dalla regola sono quelle dello stesso settore che si sono mosse più vicine nel periodo, con correlazione dei rendimenti tra 0.61 e 0.88; la coppia aggiunta per scelta (KO-PEP) ha invece una correlazione di 0.68 e un posto in classifica molto più basso.
- **Test di cointegrazione** (canale di confronto): tra le 10 coppie scelte, quelle con p-value sotto 0.05 sono 1, e quelle che passano l'intero imbuto della cointegrazione (con stabilità e correzione FDR) sono 0. Il legame statistico sul giornaliero e su 5 anni è quindi debole: i prezzi si muovono vicini, ma non c'è una forte prova di ritorno alla media.
- **Robustezza al periodo**: ripetendo la stessa regola su 10, 15 e 20 anni, 3 delle 10 coppie della lista compaiono tra le 9 scelte in almeno 2 delle 4 finestre (TFC-USB, BAC-PNC, MCD-YUM). Le coppie più "pulite" dei periodi lunghi (per esempio le classiche dell'energia) non sono state aggiunte alla lista: sono un'informazione per il report, non una modifica della selezione a posteriori.
- **Coppie "da manuale"**: di quelle che avevamo indicato in anticipo, la regola ne seleziona 3 (V-MA, HD-LOW, MCD-YUM), e ne abbiamo aggiunta una per scelta (KO-PEP). Le altre si piazzano più in basso nella classifica per SSD o hanno correlazione troppo bassa: LMT-NOC al posto 65 su 2.390; PG-CL al posto 118 su 2.390; GS-MS al posto 204 su 2.390.

### Cosa ci aspettiamo e cosa verificheremo
- La selezione è un **elenco di candidate**: la prova che le coppie funzionano arriva solo dall'addestramento e dal test sui dati intraday, usato una sola volta.
- Half-life e attraversamenti sul giornaliero misurano giorni, mentre i nostri trade durano ore: vanno ricalcolati sull'intraday.
- Le coppie con più segnalazioni nella colonna "Da guardare" (KO-PEP) sono quelle con più rischi; PYPL-XYZ, per l'ampiezza dello spread e la correlazione bassa, e ITW-OTIS, per l'half-life lunga, vanno guardate con particolare attenzione. Nel report vanno descritte con i loro rischi.
- I limiti elencati sopra (survivorship bias, GICS attuale, soglie nostre, test multipli, scelte fissate dopo aver visto i primi risultati giornalieri) vanno dichiarati nel report.
